# Data Cleaning — SPX / VIX (Variance Risk Premium Project)

Loads raw VIX and SPX price data, cleans it, aligns it on a common trading calendar, and derives the base series needed for the variance risk premium (VRP) project:

- `spx_close` — cleaned SPX close price
- `spx_log_return` — daily log return of SPX
- `vix_close` — cleaned VIX close (already an annualized % vol)

In [11]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


DATA_DIR = Path("..") / "data"
RAW_PATH = DATA_DIR / "raw_market_data.csv"
CLEAN_PATH = DATA_DIR / "clean_market_data.csv"

SPX_TICKER = "^GSPC"
VIX_TICKER = "^VIX"

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Fetch

Pull daily SPX and VIX close prices via yfinance

In [12]:
def fetch_raw_data(start: str = "2005-01-01", end: str | None = None) -> pd.DataFrame:
    """
    Pull daily SPX and VIX close prices via yfinance.
    """
    try:
        import yfinance as yf
    except ImportError as e:
        raise ImportError(
            "yfinance is required to fetch data. Install with: pip install yfinance"
        ) from e

    spx = yf.download(SPX_TICKER, start=start, end=end, progress=False)[["Close"]]
    spx.columns = ["spx_close"]

    vix = yf.download(VIX_TICKER, start=start, end=end, progress=False)[["Close"]]
    vix.columns = ["vix_close"]

    raw = spx.join(vix, how="outer")
    raw.index.name = "date"

    raw.to_csv(RAW_PATH)
    print(f"Saved raw data -> {RAW_PATH}  ({len(raw)} rows)")
    return raw

## 2. Clean


In [13]:
def clean_data(raw: pd.DataFrame, max_ffill_days: int = 3) -> pd.DataFrame:
    """
    Clean the raw SPX/VIX frame.

    Returns a tidy DataFrame indexed by date with columns:
        spx_close, vix_close, spx_log_return
    """
    df = raw.copy()

    #index hygiene
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")]

    # drop fully-empty rows
    df = df.dropna(how="all")

    #fix non-positive prices -> NaN 
    for col in ["spx_close", "vix_close"]:
        if col in df.columns:
            df.loc[df[col] <= 0, col] = np.nan

    # limited forward-fill for short gaps
    df[["spx_close", "vix_close"]] = df[["spx_close", "vix_close"]].ffill(
        limit=max_ffill_days
    )

    #drop rows that still have missing core data (real gaps/outages)
    before = len(df)
    df = df.dropna(subset=["spx_close", "vix_close"])
    dropped = before - len(df)
    if dropped:
        print(f"Dropped {dropped} rows with unrecoverable gaps (> {max_ffill_days} days).")

    #VIX outside a plausible range is a data error
    df = df[(df["vix_close"] > 1) & (df["vix_close"] < 200)]

    # derived columns 
    df["spx_log_return"] = np.log(df["spx_close"] / df["spx_close"].shift(1))

    df = df.dropna(subset=["spx_log_return"])

    return df

## 3. Public entry point



In [14]:
def load_clean_data(
    start: str = "2005-01-01",
    end: str | None = None,
    force_refresh: bool = False,
) -> pd.DataFrame:
    """
    Convenience loader used by the rest of the project.
    """
    if CLEAN_PATH.exists() and not force_refresh:
        df = pd.read_csv(CLEAN_PATH, index_col="date", parse_dates=True)
        return df

    raw = fetch_raw_data(start=start, end=end)
    clean = clean_data(raw)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    clean.to_csv(CLEAN_PATH)
    print(f"Saved clean data -> {CLEAN_PATH}  ({len(clean)} rows)")
    return clean

## 4. Run it

In [15]:
force = False  
data = load_clean_data(force_refresh=force)
print(data.head())
print(data.tail())
print(f"\nRows: {len(data)}  |  Date range: {data.index.min().date()} -> {data.index.max().date()}")

              spx_close  vix_close  spx_log_return
date                                              
2005-01-04  1188.050049      13.98       -0.011740
2005-01-05  1183.739990      14.09       -0.003634
2005-01-06  1187.890015      13.58        0.003500
2005-01-07  1186.189941      13.49       -0.001432
2005-01-10  1190.250000      13.23        0.003417
              spx_close  vix_close  spx_log_return
date                                              
2026-08-13  7798.990234      14.63        0.006495
2026-08-14  7785.759766      14.25       -0.001698
2026-08-17  7745.060059      15.19       -0.005241
2026-08-18  7691.759766      15.84       -0.006906
2026-08-19  7707.979980      14.89        0.002107

Rows: 5441  |  Date range: 2005-01-04 -> 2026-08-19
